
# 07 — Pre-POSet EDA Checks  
## Final 5-Variable Structural Snapshot

This notebook performs the final pre-POSet checks before constructing the partial order.

It runs after:

```text
06_Master_Dataset_Build_2002_5Var.ipynb
```

## Final decisions reflected here

- Final POSet ordering variables = **5**.
- WGI/governance = contextual/sensitivity only, not final ordering.
- GDP volatility/stability = diagnostic/sensitivity only, not final ordering.
- GDP recovery = validation outcome only, not final ordering.
- POSet temporal design = two **single pre-shock cross-sections**:
  - 2007 for the Global Financial Crisis
  - 2019 for the COVID shock
- Main discrete profiling uses **5 ordinal levels**.

## Why this notebook exists

Before constructing the POSet, we need to verify:

1. final complete-case sample size by shock;
2. missingness and coverage;
3. direction-aligned score ranges;
4. redundancy/correlation among the five final variables;
5. whether 5 ordinal levels are defensible;
6. whether the final variables remain interpretable and distinct;
7. report-ready methodological notes for scope, validation, and profiling choices.

## Main outputs

- `baseline_complete_case_sample_by_shock.csv`
- `candidate_variable_scorecard.csv`
- `candidate_variable_redundancy_pairs.csv`
- `candidate_variable_sets.csv`
- `discretization_level_diagnostic.csv`
- `final5_correlation_matrix_long.csv`
- `recommended_analysis_sample_by_shock.csv`
- `report_ready_pre_poset_eda_notes.csv`


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

MASTER_DIR = PROJECT_ROOT / "Data" / "Processed" / "Master_Dataset"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Pre_POSet_EDA"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "07_Pre_POSet_EDA_Checks"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "Pre_POSet_EDA"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Master folder:", MASTER_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_191546
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Master folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Master_Dataset
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\Pre_POSet_EDA


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

STRUCTURAL_SNAPSHOT_FILE = MASTER_DIR / "structural_snapshot_final5_2007_2019.csv"
COMPLETE_CASE_FILE = MASTER_DIR / "structural_snapshot_final5_complete_cases.csv"
SCORE_MAP_FILE = MASTER_DIR / "final5_ordering_variable_score_map.csv"
SCOPE_NOTE_FILE = MASTER_DIR / "report_ready_scope_and_measure_note.csv"

FINAL_5_SCORE_COLUMNS = [
    "debt_capacity_score_0_100",
    "employment_strength_score_0_100",
    "rd_intensity_score_0_100",
    "human_capital_tertiary_score_0_100",
    "energy_security_score_0_100",
]

FINAL_5_CONCEPT_LABELS = {
    "debt_capacity_score_0_100": "Debt capacity",
    "employment_strength_score_0_100": "Employment strength",
    "rd_intensity_score_0_100": "R&D intensity",
    "human_capital_tertiary_score_0_100": "Human capital",
    "energy_security_score_0_100": "Energy security",
}

FINAL_5_RAW_COLUMNS = [
    "public_debt_gdp_canonical",
    "unemployment_rate",
    "rd_gdp",
    "tertiary_education_25_64",
    "energy_import_dependency_raw",
]

MAIN_VARIABLE_SET_NAME = "baseline_5_variables"
MAIN_DISCRETIZATION_LEVELS = 5

DISCRETIZATION_LEVELS_TO_TEST = [3, 4, 5]

CORRELATION_REVIEW_THRESHOLD = 0.70
CORRELATION_HIGH_THRESHOLD = 0.80

REQUIRED_FILES = [STRUCTURAL_SNAPSHOT_FILE, COMPLETE_CASE_FILE, SCORE_MAP_FILE]

for path in REQUIRED_FILES:
    if not path.exists():
        raise FileNotFoundError(f"Required input missing: {path}")

print("Main variable set:", MAIN_VARIABLE_SET_NAME)
print("Main discretization levels:", MAIN_DISCRETIZATION_LEVELS)
print("Final 5 score columns:")
for c in FINAL_5_SCORE_COLUMNS:
    print("-", c)


Main variable set: baseline_5_variables
Main discretization levels: 5
Final 5 score columns:
- debt_capacity_score_0_100
- employment_strength_score_0_100
- rd_intensity_score_0_100
- human_capital_tertiary_score_0_100
- energy_security_score_0_100


In [3]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


In [4]:

# ------------------------------------------------------
# LOAD MASTER STRUCTURAL SNAPSHOT
# ------------------------------------------------------

snapshot = pd.read_csv(STRUCTURAL_SNAPSHOT_FILE)
complete_cases = pd.read_csv(COMPLETE_CASE_FILE)
score_map = pd.read_csv(SCORE_MAP_FILE)

for df_ in [snapshot, complete_cases]:
    df_["Country"] = df_["Country"].astype(str).str.strip().str.upper()
    df_["shock_id"] = df_["shock_id"].astype(str)

for col in FINAL_5_SCORE_COLUMNS:
    if col not in snapshot.columns:
        raise ValueError(f"Missing final score column in structural snapshot: {col}")
    if col not in complete_cases.columns:
        raise ValueError(f"Missing final score column in complete-case snapshot: {col}")

snapshot["baseline_year"] = pd.to_numeric(snapshot["baseline_year"], errors="coerce").astype(int)
complete_cases["baseline_year"] = pd.to_numeric(complete_cases["baseline_year"], errors="coerce").astype(int)

print("Snapshot rows:", len(snapshot))
print("Complete-case rows:", len(complete_cases))
print("Complete-case countries by shock:")
display(complete_cases.groupby(["shock_id", "baseline_year"])["Country"].nunique().reset_index(name="countries"))
display(score_map)


Snapshot rows: 313
Complete-case rows: 60
Complete-case countries by shock:


,shock_id,baseline_year,countries
0,2007,2007,25
1,2019,2019,35


,raw_variable,score_column,concept,direction,report_note,is_final_ordering_variable
0,public_debt_gdp_canonical,debt_capacity_score_0_100,Debt capacity,lower_is_better,Lower public debt/GDP is interpreted as greate...,True
1,unemployment_rate,employment_strength_score_0_100,Employment strength,lower_is_better,Lower unemployment is interpreted as greater l...,True
2,rd_gdp,rd_intensity_score_0_100,R&D intensity,higher_is_better,Higher R&D expenditure as a percentage of GDP ...,True
3,tertiary_education_25_64,human_capital_tertiary_score_0_100,Human capital,higher_is_better,"Higher adult tertiary educational attainment, ...",True
4,energy_import_dependency_raw,energy_security_score_0_100,Energy security,lower_is_better,Lower net energy import dependency is interpre...,True


In [5]:

# ------------------------------------------------------
# BASELINE COMPLETE-CASE SAMPLE BY SHOCK
# ------------------------------------------------------

baseline_complete_case_sample_by_shock = (
    complete_cases
    .groupby(["shock_id", "shock_label", "baseline_year"])
    .agg(
        countries=("Country", "nunique"),
        mean_final5_score_variables_available=("final5_score_variables_available", "mean"),
        min_final5_score_variables_available=("final5_score_variables_available", "min"),
        max_final5_score_variables_available=("final5_score_variables_available", "max"),
    )
    .reset_index()
    .sort_values("shock_id")
)

recommended_analysis_sample_by_shock = complete_cases[
    ["Country", "shock_id", "shock_label", "baseline_year"] + FINAL_5_SCORE_COLUMNS
].copy()

recommended_analysis_sample_by_shock["main_variable_set"] = MAIN_VARIABLE_SET_NAME
recommended_analysis_sample_by_shock["analysis_sample_role"] = "main_poset_complete_case_sample"
recommended_analysis_sample_by_shock["poset_temporal_design"] = "single_pre_shock_cross_section"

save_table(
    baseline_complete_case_sample_by_shock,
    "baseline_complete_case_sample_by_shock.csv",
    "Baseline complete-case sample by shock",
    "Complete-case country counts for the final 5-variable POSet snapshots.",
)

save_table(
    recommended_analysis_sample_by_shock,
    "recommended_analysis_sample_by_shock.csv",
    "Recommended analysis sample by shock",
    "Country-level complete-case sample to use in the main POSet notebooks.",
)

display(baseline_complete_case_sample_by_shock)
display(recommended_analysis_sample_by_shock.head())


Saved: baseline_complete_case_sample_by_shock.csv
Saved: recommended_analysis_sample_by_shock.csv


,shock_id,shock_label,baseline_year,countries,mean_final5_score_variables_available,min_final5_score_variables_available,max_final5_score_variables_available
0,2007,Global Financial Crisis,2007,25,5.0000,5,5
1,2019,COVID shock,2019,35,5.0000,5,5


,Country,shock_id,shock_label,baseline_year,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,main_variable_set,analysis_sample_role,poset_temporal_design
0,AUT,2007,Global Financial Crisis,2007,38.5303,80.6046,68.7303,33.2757,22.6201,baseline_5_variables,main_poset_complete_case_sample,single_pre_shock_cross_section
1,BEL,2007,Global Financial Crisis,2007,16.9811,46.5777,48.5360,53.5113,9.6139,baseline_5_variables,main_poset_complete_case_sample,single_pre_shock_cross_section
2,CAN,2007,Global Financial Crisis,2007,64.9865,63.6697,50.3901,100.0000,100.0000,baseline_5_variables,main_poset_complete_case_sample,single_pre_shock_cross_section
3,CZE,2007,Global Financial Crisis,2007,76.7627,74.6107,29.3904,0.4466,50.5562,baseline_5_variables,main_poset_complete_case_sample,single_pre_shock_cross_section
4,DEU,2007,Global Financial Crisis,2007,40.6157,31.1478,68.2315,30.9850,28.5833,baseline_5_variables,main_poset_complete_case_sample,single_pre_shock_cross_section


In [6]:

# ------------------------------------------------------
# FINAL 5 VARIABLE SCORECARD
# ------------------------------------------------------

scorecard_rows = []

for score_col in FINAL_5_SCORE_COLUMNS:
    concept = FINAL_5_CONCEPT_LABELS[score_col]

    coverage_by_shock = (
        complete_cases
        .groupby("shock_id")[score_col]
        .agg(
            non_missing="count",
            min_score="min",
            median_score="median",
            max_score="max",
        )
        .reset_index()
    )

    # Correlation risk: maximum absolute correlation with any other final variable.
    corr_values = []
    for shock_id, group in complete_cases.groupby("shock_id"):
        corr_matrix = group[FINAL_5_SCORE_COLUMNS].corr()
        if score_col in corr_matrix.columns:
            others = [c for c in FINAL_5_SCORE_COLUMNS if c != score_col]
            max_abs_corr = corr_matrix.loc[score_col, others].abs().max()
        else:
            max_abs_corr = np.nan
        corr_values.append(max_abs_corr)

    max_abs_corr_overall = np.nanmax(corr_values) if len(corr_values) else np.nan

    if pd.isna(max_abs_corr_overall):
        redundancy_risk = "unknown"
    elif max_abs_corr_overall >= CORRELATION_HIGH_THRESHOLD:
        redundancy_risk = "high_review"
    elif max_abs_corr_overall >= CORRELATION_REVIEW_THRESHOLD:
        redundancy_risk = "moderate_review"
    else:
        redundancy_risk = "acceptable"

    scorecard_rows.append({
        "score_column": score_col,
        "concept": concept,
        "main_variable_set": MAIN_VARIABLE_SET_NAME,
        "is_final_ordering_variable": True,
        "coverage_2007_complete_cases": int(coverage_by_shock.loc[coverage_by_shock["shock_id"].eq("2007"), "non_missing"].iloc[0]) if (coverage_by_shock["shock_id"].eq("2007")).any() else 0,
        "coverage_2019_complete_cases": int(coverage_by_shock.loc[coverage_by_shock["shock_id"].eq("2019"), "non_missing"].iloc[0]) if (coverage_by_shock["shock_id"].eq("2019")).any() else 0,
        "max_abs_correlation_with_other_final_variables": max_abs_corr_overall,
        "correlation_redundancy_risk": redundancy_risk,
        "keep_drop_decision": "keep",
        "reason": "Part of the final five conceptually distinct resilience-capacity dimensions.",
    })

candidate_variable_scorecard = pd.DataFrame(scorecard_rows)

save_table(
    candidate_variable_scorecard,
    "candidate_variable_scorecard.csv",
    "Candidate variable scorecard",
    "Final 5-variable scorecard with coverage and redundancy checks.",
)

display(candidate_variable_scorecard)


Saved: candidate_variable_scorecard.csv


,score_column,concept,main_variable_set,is_final_ordering_variable,coverage_2007_complete_cases,coverage_2019_complete_cases,max_abs_correlation_with_other_final_variables,correlation_redundancy_risk,keep_drop_decision,reason
0,debt_capacity_score_0_100,Debt capacity,baseline_5_variables,True,25,35,0.3374,acceptable,keep,Part of the final five conceptually distinct r...
1,employment_strength_score_0_100,Employment strength,baseline_5_variables,True,25,35,0.4085,acceptable,keep,Part of the final five conceptually distinct r...
2,rd_intensity_score_0_100,R&D intensity,baseline_5_variables,True,25,35,0.5374,acceptable,keep,Part of the final five conceptually distinct r...
3,human_capital_tertiary_score_0_100,Human capital,baseline_5_variables,True,25,35,0.5374,acceptable,keep,Part of the final five conceptually distinct r...
4,energy_security_score_0_100,Energy security,baseline_5_variables,True,25,35,0.4731,acceptable,keep,Part of the final five conceptually distinct r...


In [7]:

# ------------------------------------------------------
# CORRELATION AND REDUNDANCY CHECKS
# ------------------------------------------------------

corr_long_rows = []
redundancy_rows = []

for shock_id, group in complete_cases.groupby("shock_id"):
    corr = group[FINAL_5_SCORE_COLUMNS].corr()

    # Save wide correlation matrix by shock too.
    corr_wide = corr.reset_index().rename(columns={"index": "variable"})
    save_table(
        corr_wide,
        f"final5_correlation_matrix_shock_{shock_id}.csv",
        f"Final 5 correlation matrix shock {shock_id}",
        f"Wide correlation matrix for final 5 score variables in shock {shock_id}.",
    )

    for var1 in FINAL_5_SCORE_COLUMNS:
        for var2 in FINAL_5_SCORE_COLUMNS:
            corr_value = corr.loc[var1, var2]
            corr_long_rows.append({
                "shock_id": shock_id,
                "variable_1": var1,
                "variable_2": var2,
                "correlation": corr_value,
                "abs_correlation": abs(corr_value) if pd.notna(corr_value) else np.nan,
            })

    for i, var1 in enumerate(FINAL_5_SCORE_COLUMNS):
        for var2 in FINAL_5_SCORE_COLUMNS[i + 1:]:
            corr_value = corr.loc[var1, var2]
            abs_corr = abs(corr_value) if pd.notna(corr_value) else np.nan

            if pd.isna(abs_corr):
                flag = "unknown"
            elif abs_corr >= CORRELATION_HIGH_THRESHOLD:
                flag = "high_redundancy_review"
            elif abs_corr >= CORRELATION_REVIEW_THRESHOLD:
                flag = "moderate_redundancy_review"
            else:
                flag = "acceptable"

            redundancy_rows.append({
                "shock_id": shock_id,
                "variable_1": var1,
                "variable_2": var2,
                "correlation": corr_value,
                "abs_correlation": abs_corr,
                "redundancy_flag": flag,
                "decision": "review_only_no_automatic_drop" if flag != "acceptable" else "keep_pair",
            })

final5_correlation_matrix_long = pd.DataFrame(corr_long_rows)
candidate_variable_redundancy_pairs = pd.DataFrame(redundancy_rows)

save_table(
    final5_correlation_matrix_long,
    "final5_correlation_matrix_long.csv",
    "Final 5 correlation matrix long",
    "Long-format correlation matrix for final 5 score variables by shock.",
)

save_table(
    candidate_variable_redundancy_pairs,
    "candidate_variable_redundancy_pairs.csv",
    "Candidate variable redundancy pairs",
    "Pairwise correlation/redundancy check for the final 5 score variables.",
)

display(candidate_variable_redundancy_pairs.sort_values(["redundancy_flag", "abs_correlation"], ascending=[True, False]))


Saved: final5_correlation_matrix_shock_2007.csv
Saved: final5_correlation_matrix_shock_2019.csv
Saved: final5_correlation_matrix_long.csv
Saved: candidate_variable_redundancy_pairs.csv


,shock_id,variable_1,variable_2,correlation,abs_correlation,redundancy_flag,decision
7,2007,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,0.5374,0.5374,acceptable,keep_pair
17,2019,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,0.5079,0.5079,acceptable,keep_pair
9,2007,human_capital_tertiary_score_0_100,energy_security_score_0_100,0.4731,0.4731,acceptable,keep_pair
5,2007,employment_strength_score_0_100,human_capital_tertiary_score_0_100,0.4085,0.4085,acceptable,keep_pair
15,2019,employment_strength_score_0_100,human_capital_tertiary_score_0_100,0.3843,0.3843,acceptable,keep_pair
14,2019,employment_strength_score_0_100,rd_intensity_score_0_100,0.3623,0.3623,acceptable,keep_pair
0,2007,debt_capacity_score_0_100,employment_strength_score_0_100,0.3374,0.3374,acceptable,keep_pair
10,2019,debt_capacity_score_0_100,employment_strength_score_0_100,0.3252,0.3252,acceptable,keep_pair
4,2007,employment_strength_score_0_100,rd_intensity_score_0_100,0.2938,0.2938,acceptable,keep_pair
8,2007,rd_intensity_score_0_100,energy_security_score_0_100,0.2763,0.2763,acceptable,keep_pair


In [8]:

# ------------------------------------------------------
# DISCRETIZATION LEVEL DIAGNOSTICS
# ------------------------------------------------------
# The main POSet profile uses 5 ordinal levels.
# This cell checks profile uniqueness and profile compression for 3, 4, and 5 levels.
#
# Higher score = better. Level 1 = lowest structural score band; Level K = highest.

def assign_quantile_levels(series, levels):
    # Rank-first avoids qcut failures with ties while preserving order.
    ranked = series.rank(method="first")
    return pd.qcut(ranked, q=levels, labels=list(range(1, levels + 1))).astype(int)

def build_profile_codes(df_in, score_columns, levels):
    out = df_in.copy()
    level_cols = []

    for col in score_columns:
        level_col = col.replace("_score_0_100", f"_level_{levels}")
        out[level_col] = assign_quantile_levels(out[col], levels)
        level_cols.append(level_col)

    out[f"profile_code_{levels}levels"] = out[level_cols].astype(str).agg("-".join, axis=1)
    out[f"profile_digit_sum_{levels}levels"] = out[level_cols].sum(axis=1)

    return out, level_cols

discretization_rows = []
profile_detail_frames = []

for shock_id, group in complete_cases.groupby("shock_id"):
    group = group.copy()
    n_countries = group["Country"].nunique()

    for levels in DISCRETIZATION_LEVELS_TO_TEST:
        profiled, level_cols = build_profile_codes(group, FINAL_5_SCORE_COLUMNS, levels)
        profile_col = f"profile_code_{levels}levels"

        unique_profiles = profiled[profile_col].nunique()
        max_countries_per_profile = profiled.groupby(profile_col)["Country"].nunique().max()
        mean_countries_per_level_bin = n_countries / levels

        discretization_rows.append({
            "shock_id": shock_id,
            "levels": levels,
            "countries": n_countries,
            "variables": len(FINAL_5_SCORE_COLUMNS),
            "unique_profiles": unique_profiles,
            "profile_uniqueness_ratio": unique_profiles / n_countries if n_countries else np.nan,
            "max_countries_per_profile": max_countries_per_profile,
            "mean_countries_per_level_bin": mean_countries_per_level_bin,
            "is_main_choice": levels == MAIN_DISCRETIZATION_LEVELS,
            "interpretation": (
                "Main choice: five levels preserve ordinal detail while remaining interpretable."
                if levels == MAIN_DISCRETIZATION_LEVELS
                else "Sensitivity/diagnostic alternative."
            ),
        })

        keep_cols = ["Country", "shock_id", "baseline_year"] + FINAL_5_SCORE_COLUMNS + level_cols + [profile_col, f"profile_digit_sum_{levels}levels"]
        temp = profiled[keep_cols].copy()
        temp["levels"] = levels
        profile_detail_frames.append(temp)

discretization_level_diagnostic = pd.DataFrame(discretization_rows)
profile_discretization_detail = pd.concat(profile_detail_frames, ignore_index=True)

main_profile_discretization_5levels = profile_discretization_detail[
    profile_discretization_detail["levels"].eq(MAIN_DISCRETIZATION_LEVELS)
].copy()

save_table(
    discretization_level_diagnostic,
    "discretization_level_diagnostic.csv",
    "Discretization level diagnostic",
    "Compares 3, 4, and 5 ordinal levels before final POSet profiling.",
)

save_table(
    profile_discretization_detail,
    "profile_discretization_detail.csv",
    "Profile discretization detail",
    "Country-level ordinal profile codes for 3, 4, and 5 levels.",
)

save_table(
    main_profile_discretization_5levels,
    "main_profile_discretization_5levels.csv",
    "Main profile discretization 5 levels",
    "Country-level 5-level ordinal profiles for the final main POSet profile construction.",
)

display(discretization_level_diagnostic)
display(main_profile_discretization_5levels.head())


Saved: discretization_level_diagnostic.csv
Saved: profile_discretization_detail.csv
Saved: main_profile_discretization_5levels.csv


,shock_id,levels,countries,variables,unique_profiles,profile_uniqueness_ratio,max_countries_per_profile,mean_countries_per_level_bin,is_main_choice,interpretation
0,2007,3,25,5,22,0.8800,2,8.3333,False,Sensitivity/diagnostic alternative.
1,2007,4,25,5,25,1.0000,1,6.2500,False,Sensitivity/diagnostic alternative.
2,2007,5,25,5,25,1.0000,1,5.0000,True,Main choice: five levels preserve ordinal deta...
3,2019,3,35,5,28,0.8000,2,11.6667,False,Sensitivity/diagnostic alternative.
4,2019,4,35,5,35,1.0000,1,8.7500,False,Sensitivity/diagnostic alternative.
5,2019,5,35,5,34,0.9714,2,7.0000,True,Main choice: five levels preserve ordinal deta...


,Country,shock_id,baseline_year,debt_capacity_score_0_100,employment_strength_score_0_100,rd_intensity_score_0_100,human_capital_tertiary_score_0_100,energy_security_score_0_100,debt_capacity_level_3,employment_strength_level_3,rd_intensity_level_3,human_capital_tertiary_level_3,energy_security_level_3,profile_code_3levels,profile_digit_sum_3levels,levels,debt_capacity_level_4,employment_strength_level_4,rd_intensity_level_4,human_capital_tertiary_level_4,energy_security_level_4,profile_code_4levels,profile_digit_sum_4levels,debt_capacity_level_5,employment_strength_level_5,rd_intensity_level_5,human_capital_tertiary_level_5,energy_security_level_5,profile_code_5levels,profile_digit_sum_5levels
50,AUT,2007,2007,38.5303,80.6046,68.7303,33.2757,22.6201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0000,4.0000,5.0000,3.0000,2.0000,2-4-5-3-2,16.0000
51,BEL,2007,2007,16.9811,46.5777,48.5360,53.5113,9.6139,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0000,2.0000,4.0000,4.0000,1.0000,1-2-4-4-1,12.0000
52,CAN,2007,2007,64.9865,63.6697,50.3901,100.0000,100.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0000,3.0000,4.0000,5.0000,5.0000,3-3-4-5-5,20.0000
53,CZE,2007,2007,76.7627,74.6107,29.3904,0.4466,50.5562,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0000,3.0000,3.0000,1.0000,5.0000,4-3-3-1-5,16.0000
54,DEU,2007,2007,40.6157,31.1478,68.2315,30.9850,28.5833,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0000,1.0000,4.0000,2.0000,3.0000,2-1-4-2-3,12.0000


In [9]:

# ------------------------------------------------------
# VARIABLE SET DEFINITIONS
# ------------------------------------------------------
# These definitions will be used by downstream POSet and sensitivity notebooks.

candidate_variable_sets = pd.DataFrame([
    {
        "set_name": "baseline_5_variables",
        "set_role": "main",
        "variables": ",".join(FINAL_5_SCORE_COLUMNS),
        "n_variables": len(FINAL_5_SCORE_COLUMNS),
        "include_governance": False,
        "include_volatility": False,
        "include_recovery": False,
        "reason": "Final main POSet variable set: five conceptually distinct pre-shock structural capacity dimensions.",
    },
    {
        "set_name": "sensitivity_drop_energy_security",
        "set_role": "sensitivity",
        "variables": ",".join([v for v in FINAL_5_SCORE_COLUMNS if v != "energy_security_score_0_100"]),
        "n_variables": len(FINAL_5_SCORE_COLUMNS) - 1,
        "include_governance": False,
        "include_volatility": False,
        "include_recovery": False,
        "reason": "Sensitivity check because energy security can behave differently for exporters and importers.",
    },
    {
        "set_name": "sensitivity_drop_most_correlated_pair_member",
        "set_role": "sensitivity_placeholder",
        "variables": "to_be_selected_from_redundancy_pairs_if_needed",
        "n_variables": np.nan,
        "include_governance": False,
        "include_volatility": False,
        "include_recovery": False,
        "reason": "Optional sensitivity if redundancy diagnostics show a high-correlation pair.",
    },
])

save_table(
    candidate_variable_sets,
    "candidate_variable_sets.csv",
    "Candidate variable sets",
    "Main and sensitivity variable-set definitions for downstream POSet analysis.",
)

display(candidate_variable_sets)


Saved: candidate_variable_sets.csv


,set_name,set_role,variables,n_variables,include_governance,include_volatility,include_recovery,reason
0,baseline_5_variables,main,"debt_capacity_score_0_100,employment_strength_...",5.0000,False,False,False,Final main POSet variable set: five conceptual...
1,sensitivity_drop_energy_security,sensitivity,"debt_capacity_score_0_100,employment_strength_...",4.0000,False,False,False,Sensitivity check because energy security can ...
2,sensitivity_drop_most_correlated_pair_member,sensitivity_placeholder,to_be_selected_from_redundancy_pairs_if_needed,NaN,False,False,False,Optional sensitivity if redundancy diagnostics...


In [10]:

# ------------------------------------------------------
# VALIDATION DESIGN NOTE
# ------------------------------------------------------
# The validation variables are not used in POSet construction.
# This table is report-support material for Francesco's point about validation.

validation_design_table = pd.DataFrame([
    {
        "validation_component": "GDP recovery time",
        "variable_or_output": "Recovery_2007, Recovery_2019",
        "how_used": "Compare recovery years across Pareto/frontier status, POSet layers, and structural profile groups.",
        "why_included": "Direct outcome-style measure of return to pre-shock GDP level.",
        "leakage_control": "Not used as an ordering variable.",
    },
    {
        "validation_component": "GDP growth path",
        "variable_or_output": "country_recovery_index_paths_dynamic_baseline",
        "how_used": "Interpret country-specific recovery trajectories and mismatches.",
        "why_included": "Shows whether recovery is fast, slow, or incomplete after shock.",
        "leakage_control": "Computed after structural baseline; not included in POSet scoring.",
    },
    {
        "validation_component": "Unemployment",
        "variable_or_output": "unemployment_rate",
        "how_used": "Macro validation dimension for labour-market stress/recovery.",
        "why_included": "Employment is a central socio-economic resilience outcome and capacity proxy.",
        "leakage_control": "Baseline unemployment contributes to employment strength; post-shock unemployment is only validation.",
    },
    {
        "validation_component": "Inflation",
        "variable_or_output": "inflation_cpi",
        "how_used": "Macro validation dimension for price stability after shock.",
        "why_included": "Inflation captures macroeconomic stress not captured by GDP recovery alone.",
        "leakage_control": "Not used in final POSet ordering.",
    },
    {
        "validation_component": "Public debt",
        "variable_or_output": "public_debt_gdp_canonical",
        "how_used": "Contextual validation of fiscal stress after shocks.",
        "why_included": "Debt evolution helps interpret fiscal resilience and policy space.",
        "leakage_control": "Only baseline debt is used for ordering; post-shock debt is validation/context.",
    },
    {
        "validation_component": "Productivity",
        "variable_or_output": "productivity_gdp_per_hour",
        "how_used": "Macro validation dimension for productive capacity after shock.",
        "why_included": "Productivity captures structural economic performance beyond short-term GDP growth.",
        "leakage_control": "Not used in final POSet ordering.",
    },
])

save_table(
    validation_design_table,
    "validation_design_table.csv",
    "Validation design table",
    "Report-support table describing how validation variables are used and why they are not ordering variables.",
)

display(validation_design_table)


Saved: validation_design_table.csv


,validation_component,variable_or_output,how_used,why_included,leakage_control
0,GDP recovery time,"Recovery_2007, Recovery_2019",Compare recovery years across Pareto/frontier ...,Direct outcome-style measure of return to pre-...,Not used as an ordering variable.
1,GDP growth path,country_recovery_index_paths_dynamic_baseline,Interpret country-specific recovery trajectori...,"Shows whether recovery is fast, slow, or incom...",Computed after structural baseline; not includ...
2,Unemployment,unemployment_rate,Macro validation dimension for labour-market s...,Employment is a central socio-economic resilie...,Baseline unemployment contributes to employmen...
3,Inflation,inflation_cpi,Macro validation dimension for price stability...,Inflation captures macroeconomic stress not ca...,Not used in final POSet ordering.
4,Public debt,public_debt_gdp_canonical,Contextual validation of fiscal stress after s...,Debt evolution helps interpret fiscal resilien...,Only baseline debt is used for ordering; post-...
5,Productivity,productivity_gdp_per_hour,Macro validation dimension for productive capa...,Productivity captures structural economic perf...,Not used in final POSet ordering.


In [11]:

# ------------------------------------------------------
# REPORT-READY PRE-POSET EDA NOTES
# ------------------------------------------------------

report_ready_pre_poset_eda_notes = pd.DataFrame([
    {
        "topic": "Why five variables",
        "report_text": (
            "The final POSet uses five ordering variables because they represent distinct resilience-relevant "
            "capacities: fiscal capacity, labour-market strength, innovation capacity, human capital, and energy "
            "security. Governance and volatility are retained as contextual or sensitivity information rather "
            "than final ordering variables."
        ),
    },
    {
        "topic": "Why single cross-sections",
        "report_text": (
            "The POSet is constructed on two single pre-shock cross-sections, 2007 and 2019. This design avoids "
            "mixing post-shock recovery information into the structural ordering variables and keeps the analysis "
            "focused on country capacity immediately before each shock."
        ),
    },
    {
        "topic": "Why five ordinal levels",
        "report_text": (
            "Five ordinal levels are used for the main profile construction because they provide a quintile-like "
            "middle ground between excessive compression and overly granular pseudo-ranking. With 25 complete "
            "countries in 2007 and 35 in 2019, five levels retain meaningful structural differences while keeping "
            "the profiles interpretable. Sensitivity checks compare alternative discretizations."
        ),
    },
    {
        "topic": "Why not a single scalar index",
        "report_text": (
            "The direction-aligned 0–100 scores are not averaged into a final index. They are used only to create "
            "ordered profiles and dominance relations. The POSet framework preserves incomparability instead of "
            "forcing every country into a total ranking."
        ),
    },
    {
        "topic": "Validation design",
        "report_text": (
            "Validation is performed after POSet construction by comparing frontier status, layers, and profiles "
            "against GDP recovery and additional macroeconomic outcomes. These validation variables do not enter "
            "the ordering relation."
        ),
    },
    {
        "topic": "Temporal and spatial scope",
        "report_text": (
            "The spatial unit is the country. The temporal data panel covers 2002–2024/2025 depending on the source, "
            "but the POSet itself uses the two pre-shock baseline years because resilience capacity is measured at "
            "the moment just before the relevant shock."
        ),
    },
])

methodological_decision_updates_pre_poset = pd.DataFrame([
    {
        "decision": "Main discretization uses five ordinal levels",
        "choice": "5 levels / quintile-like profiling",
        "reason": "Balances interpretability with sufficient differentiation for 25 and 35 complete-case countries.",
        "sensitivity_or_check": "Compare 3, 4, and 5 levels in sensitivity analysis.",
    },
    {
        "decision": "Use baseline_5_variables as main POSet set",
        "choice": ",".join(FINAL_5_SCORE_COLUMNS),
        "reason": "Covers five distinct structural capacity dimensions without governance redundancy.",
        "sensitivity_or_check": "Run variable-set sensitivity checks downstream.",
    },
    {
        "decision": "Keep validation variables outside ordering",
        "choice": "GDP recovery and macro indicators only for validation",
        "reason": "Prevents outcome leakage into POSet construction.",
        "sensitivity_or_check": "Validation notebook compares profiles/layers/frontiers against recovery outcomes.",
    },
])

save_table(
    report_ready_pre_poset_eda_notes,
    "report_ready_pre_poset_eda_notes.csv",
    "Report-ready Pre-POSet EDA notes",
    "Report-ready paragraphs covering five variables, cross-sections, five levels, validation design, and scope.",
)

save_table(
    methodological_decision_updates_pre_poset,
    "methodological_decision_updates_pre_poset.csv",
    "Methodological decision updates Pre-POSet",
    "Decision-log entries for discretization, variable set, and validation design.",
)

display(report_ready_pre_poset_eda_notes)
display(methodological_decision_updates_pre_poset)


Saved: report_ready_pre_poset_eda_notes.csv
Saved: methodological_decision_updates_pre_poset.csv


,topic,report_text
0,Why five variables,The final POSet uses five ordering variables b...
1,Why single cross-sections,The POSet is constructed on two single pre-sho...
2,Why five ordinal levels,Five ordinal levels are used for the main prof...
3,Why not a single scalar index,The direction-aligned 0–100 scores are not ave...
4,Validation design,Validation is performed after POSet constructi...
5,Temporal and spatial scope,The spatial unit is the country. The temporal ...


,decision,choice,reason,sensitivity_or_check
0,Main discretization uses five ordinal levels,5 levels / quintile-like profiling,Balances interpretability with sufficient diff...,"Compare 3, 4, and 5 levels in sensitivity anal..."
1,Use baseline_5_variables as main POSet set,"debt_capacity_score_0_100,employment_strength_...",Covers five distinct structural capacity dimen...,Run variable-set sensitivity checks downstream.
2,Keep validation variables outside ordering,GDP recovery and macro indicators only for val...,Prevents outcome leakage into POSet construction.,Validation notebook compares profiles/layers/f...


In [12]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "07_Pre_POSet_EDA_Checks_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    baseline_complete_case_sample_by_shock.to_excel(writer, sheet_name="sample_by_shock", index=False)
    candidate_variable_scorecard.to_excel(writer, sheet_name="variable_scorecard", index=False)
    candidate_variable_redundancy_pairs.to_excel(writer, sheet_name="redundancy_pairs", index=False)
    final5_correlation_matrix_long.to_excel(writer, sheet_name="correlation_long", index=False)
    discretization_level_diagnostic.to_excel(writer, sheet_name="discretization_diag", index=False)
    main_profile_discretization_5levels.to_excel(writer, sheet_name="main_5level_profiles", index=False)
    candidate_variable_sets.to_excel(writer, sheet_name="variable_sets", index=False)
    recommended_analysis_sample_by_shock.to_excel(writer, sheet_name="analysis_sample", index=False)
    validation_design_table.to_excel(writer, sheet_name="validation_design", index=False)
    report_ready_pre_poset_eda_notes.to_excel(writer, sheet_name="report_notes", index=False)
    methodological_decision_updates_pre_poset.to_excel(writer, sheet_name="decision_updates", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\07_Pre_POSet_EDA_Checks_Audit.xlsx


In [13]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "pre_poset_eda_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "pre_poset_eda_table_inventory.csv", index=False)

expected_outputs = [
    "baseline_complete_case_sample_by_shock.csv",
    "recommended_analysis_sample_by_shock.csv",
    "candidate_variable_scorecard.csv",
    "candidate_variable_redundancy_pairs.csv",
    "final5_correlation_matrix_long.csv",
    "discretization_level_diagnostic.csv",
    "profile_discretization_detail.csv",
    "main_profile_discretization_5levels.csv",
    "candidate_variable_sets.csv",
    "validation_design_table.csv",
    "report_ready_pre_poset_eda_notes.csv",
    "methodological_decision_updates_pre_poset.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("07 PRE-POSET EDA CHECKS COMPLETE")
print("=" * 90)

display(output_check)

print("\nKey outputs to inspect/send:")
print("- 07_Pre_POSet_EDA_Checks_Audit.xlsx")
print("- candidate_variable_scorecard.csv")
print("- candidate_variable_redundancy_pairs.csv")
print("- discretization_level_diagnostic.csv")
print("- report_ready_pre_poset_eda_notes.csv")

print("\nNext notebook:")
print("08_Profile_POSet_Main_2002_5Var.ipynb")


07 PRE-POSET EDA CHECKS COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,baseline_complete_case_sample_by_shock.csv,True,True,True
1,recommended_analysis_sample_by_shock.csv,True,True,True
2,candidate_variable_scorecard.csv,True,True,True
3,candidate_variable_redundancy_pairs.csv,True,True,True
4,final5_correlation_matrix_long.csv,True,True,True
5,discretization_level_diagnostic.csv,True,True,True
6,profile_discretization_detail.csv,True,True,True
7,main_profile_discretization_5levels.csv,True,True,True
8,candidate_variable_sets.csv,True,True,True
9,validation_design_table.csv,True,True,True



Key outputs to inspect/send:
- 07_Pre_POSet_EDA_Checks_Audit.xlsx
- candidate_variable_scorecard.csv
- candidate_variable_redundancy_pairs.csv
- discretization_level_diagnostic.csv
- report_ready_pre_poset_eda_notes.csv

Next notebook:
08_Profile_POSet_Main_2002_5Var.ipynb
